In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import load_img, img_to_array, ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# --- 1. Load Dataset ---
IMAGE_DIR = "images"
ANNOT_DIR = "annotations"
IMG_SIZE = (224, 224)

print("📦 Starting data loading process...")
X, y_expr, y_valence, y_arousal = [], [], [], []

file_prefixes = sorted(list(set([f.split('_')[0] for f in os.listdir(ANNOT_DIR)])))

for prefix in file_prefixes:
    try:
        img_path = os.path.join(IMAGE_DIR, f"{prefix}.jpg")
        expr_path = os.path.join(ANNOT_DIR, f"{prefix}_exp.npy")
        val_path = os.path.join(ANNOT_DIR, f"{prefix}_val.npy")
        aro_path = os.path.join(ANNOT_DIR, f"{prefix}_aro.npy")

        if not all(os.path.exists(p) for p in [img_path, expr_path, val_path, aro_path]):
            continue

        valence = np.load(val_path)
        arousal = np.load(aro_path)

        if valence == -2 or arousal == -2:
            continue

        img = load_img(img_path, target_size=IMG_SIZE)
        img_array = img_to_array(img) / 255.0

        X.append(img_array)
        y_expr.append(np.load(expr_path))
        y_valence.append(valence)
        y_arousal.append(arousal)

    except Exception as e:
        print(f"⚠️ Skipping {prefix} due to error: {e}")

X = np.array(X, dtype="float32")
y_expr = np.array(y_expr, dtype="int")
y_valence = np.array(y_valence, dtype="float32")
y_arousal = np.array(y_arousal, dtype="float32")
y_va = np.stack((y_valence, y_arousal), axis=1)

print("\n✅ Dataset Loaded:")
print(f"Images: {X.shape}, Expressions: {y_expr.shape}, Valence-Arousal: {y_va.shape}")

# --- 2. Split Data ---
X_train, X_val, y_expr_train, y_expr_val, y_va_train, y_va_val = train_test_split(
    X, y_expr, y_va,
    test_size=0.2,
    random_state=42,
    stratify=y_expr
)

print("\n✅ Data Split:")
print(f"X_train: {X_train.shape}, X_val: {X_val.shape}")
print(f"y_expr_train: {y_expr_train.shape}, y_va_train: {y_va_train.shape}")

# --- 3. Data Augmentation ---
datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True
)
datagen.fit(X_train)

# --- 4. Custom Generator for Multi-output ---
class MultiOutputDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, image_data_generator, X, y_dict, batch_size=32, **kwargs):
        super().__init__(**kwargs)
        self.image_data_generator = image_data_generator
        self.X = X
        self.y_dict = y_dict
        self.batch_size = batch_size
        self.generator = self.image_data_generator.flow(
            X, y_dict["expression_output"], batch_size=batch_size, shuffle=True
        )
        self.indexes = np.arange(len(X))

    def __len__(self):
        return int(np.ceil(len(self.X) / self.batch_size))

    def __getitem__(self, index):
        batch_X, batch_y_expr = self.generator[index]
        batch_indices = self.generator.index_array[
            index * self.batch_size:(index + 1) * self.batch_size
        ]
        batch_y_va = self.y_dict["va_output"][batch_indices]
        return batch_X, {
            "expression_output": batch_y_expr,
            "va_output": batch_y_va
        }

# --- 5. Build Model ---
base_model = VGG16(weights="imagenet", include_top=False, input_shape=(224, 224, 3))

for layer in base_model.layers[:-8]:
    layer.trainable = False
for layer in base_model.layers[-8:]:
    layer.trainable = True

x = Flatten()(base_model.output)
x = Dense(512, activation="relu")(x)
x = Dropout(0.5)(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.3)(x)

expr_output = Dense(8, activation="softmax", name="expression_output")(x)
va_output = Dense(2, activation="linear", name="va_output")(x)

model = Model(inputs=base_model.input, outputs=[expr_output, va_output])

# --- 6. Compile Model ---
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss={
        "expression_output": "sparse_categorical_crossentropy",
        "va_output": "mean_squared_error"
    },
    loss_weights={
        "expression_output": 1.0,
        "va_output": 0.5
    },
    metrics={
        "expression_output": "accuracy",
        "va_output": "mae"
    }
)

model.summary()

# --- 7. Callbacks ---
callbacks = [
    EarlyStopping(monitor="val_expression_output_accuracy", mode="max", patience=5, restore_best_weights=True),
    ModelCheckpoint("best_model.h5", monitor="val_expression_output_accuracy", save_best_only=True, mode="max")
]

# --- 8. Training ---
train_generator = MultiOutputDataGenerator(datagen, X_train, {
    "expression_output": y_expr_train,
    "va_output": y_va_train
}, batch_size=32)

history = model.fit(
    train_generator,
    validation_data=(X_val, {
        "expression_output": y_expr_val,
        "va_output": y_va_val
    }),
    epochs=30,
    callbacks=callbacks
)

# --- 9. Evaluation ---
print("\n✅ Evaluating model on validation set:")
results = model.evaluate(X_val, {
    "expression_output": y_expr_val,
    "va_output": y_va_val
})

print("\n📊 Final Validation Results:")
print(f"Total Loss: {results[0]:.4f}")
print(f"Expression Loss: {results[1]:.4f}")
print(f"Valence-Arousal Loss: {results[2]:.4f}")
print(f"Expression Accuracy: {results[3]*100:.2f}%")
print(f"Valence-Arousal MAE: {results[4]:.4f}")


📦 Starting data loading process...

✅ Dataset Loaded:
Images: (3999, 224, 224, 3), Expressions: (3999,), Valence-Arousal: (3999, 2)

✅ Data Split:
X_train: (3199, 224, 224, 3), X_val: (800, 224, 224, 3)
y_expr_train: (3199,), y_va_train: (3199, 2)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1        │ (None, 224, 224,  │      1,792 │ input_layer[0][0] │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2        │ (None, 224, 224,  │     36,928 │ block1_conv1[0][… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_pool         │ (None, 112, 112,  │          0 │ block1_conv2[0][… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_conv1        │ (None, 112, 112,  │     73,856 │ block1_pool[0][0] │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_conv2        │ (None, 112, 112,  │    147,584 │ block2_conv1[0][… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_pool         │ (None, 56, 56,    │          0 │ block2_conv2[0][… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_conv1        │ (None, 56, 56,    │    295,168 │ block2_pool[0][0] │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_conv2        │ (None, 56, 56,    │    590,080 │ block3_conv1[0][… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_conv3        │ (None, 56, 56,    │    590,080 │ block3_conv2[0][… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_pool         │ (None, 28, 28,    │          0 │ block3_conv3[0][… │
│ (MaxPooling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block4_conv1        │ (None, 28, 28,    │  1,180,160 │ block3_pool[0][0] │
│ (Conv2D)            │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block4_conv2        │ (None, 28, 28,    │  2,359,808 │ block4_conv1[0][… │
│ (Conv2D)            │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block4_conv3        │ (None, 28, 28,    │  2,359,808 │ block4_conv2[0][… │
│ (Conv2D)            │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block4_pool         │ (None, 14, 14,    │          0 │ block4_conv3[0][… │
│ (MaxPooling2D)      │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block5_conv1        │ (None, 14, 14,    │  2,359,808 │ block4_pool[0][0] │
│ (Conv2D)            │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block5_conv2        │ (None, 14, 14,    │  2,359,808 │ block5_conv1[0][

 Total params: 27,694,154 (105.64 MB)

 Trainable params: 25,958,666 (99.02 MB)

...

100/100 ━━━━━━━━━━━━━━━━━━━━ 64s 637ms/step - expression_output_accuracy: 0.3191 - expression_output_loss: 1.7542 - loss: 1.8508 - va_output_loss: 0.1931 - va_output_mae: 0.3564 - val_expression_output_accuracy: 0.3250 - val_expression_output_loss: 1.6971 - val_loss: 1.7722 - val_va_output_loss: 0.1502 - val_va_output_mae: 0.3201
Epoch 20/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 427ms/step - expression_output_accuracy: 0.3239 - expression_output_loss: 1.7104 - loss: 1.8057 - va_output_loss: 0.1905 - va_output_mae: 0.3576

100/100 ━━━━━━━━━━━━━━━━━━━━ 54s 535ms/step - expression_output_accuracy: 0.3239 - expression_output_loss: 1.7104 - loss: 1.8056 - va_output_loss: 0.1905 - va_output_mae: 0.3576 - val_expression_output_accuracy: 0.3613 - val_expression_output_loss: 1.6385 - val_loss: 1.7109 - val_va_output_loss: 0.1448 - val_va_output_mae: 0.3096
Epoch 21/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 48s 481ms/step - expression_output_accuracy: 0.3356 - expression_output_loss: 1.6656 - loss: 1.7647 - va_output_loss: 0.1982 - va_output_mae: 0.3616 - val_expression_output_accuracy: 0.3338 - val_expression_output_loss: 1.6696 - val_loss: 1.7432 - val_va_output_loss: 0.1471 - val_va_output_mae: 0.3156
Epoch 22/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 47s 473ms/step - expression_output_accuracy: 0.3574 - expression_output_loss: 1.6311 - loss: 1.7228 - va_output_loss: 0.1835 - va_output_mae: 0.3482 - val_expression_output_accuracy: 0.3438 - val_expression_output_loss: 1.6658 - val_loss: 1.7386 - val_va_output_loss: 0.1457 - val_va

100/100 ━━━━━━━━━━━━━━━━━━━━ 53s 529ms/step - expression_output_accuracy: 0.3713 - expression_output_loss: 1.5860 - loss: 1.6737 - va_output_loss: 0.1754 - va_output_mae: 0.3395 - val_expression_output_accuracy: 0.3738 - val_expression_output_loss: 1.6221 - val_loss: 1.6910 - val_va_output_loss: 0.1378 - val_va_output_mae: 0.3017
Epoch 24/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 48s 479ms/step - expression_output_accuracy: 0.3969 - expression_output_loss: 1.5574 - loss: 1.6464 - va_output_loss: 0.1779 - va_output_mae: 0.3452 - val_expression_output_accuracy: 0.3562 - val_expression_output_loss: 1.6128 - val_loss: 1.6839 - val_va_output_loss: 0.1423 - val_va_output_mae: 0.3078
Epoch 25/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 48s 479ms/step - expression_output_accuracy: 0.3849 - expression_output_loss: 1.5382 - loss: 1.6272 - va_output_loss: 0.1780 - va_output_mae: 0.3453 - val_expression_output_accuracy: 0.3688 - val_expression_output_loss: 1.5951 - val_loss: 1.6651 - val_va_output_loss: 0.1401 - val_va

100/100 ━━━━━━━━━━━━━━━━━━━━ 65s 648ms/step - expression_output_accuracy: 0.4188 - expression_output_loss: 1.4866 - loss: 1.5736 - va_output_loss: 0.1740 - va_output_mae: 0.3377 - val_expression_output_accuracy: 0.4000 - val_expression_output_loss: 1.5907 - val_loss: 1.6589 - val_va_output_loss: 0.1365 - val_va_output_mae: 0.2936
Epoch 28/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 428ms/step - expression_output_accuracy: 0.4264 - expression_output_loss: 1.4447 - loss: 1.5266 - va_output_loss: 0.1637 - va_output_mae: 0.3292

100/100 ━━━━━━━━━━━━━━━━━━━━ 64s 638ms/step - expression_output_accuracy: 0.4265 - expression_output_loss: 1.4446 - loss: 1.5265 - va_output_loss: 0.1637 - va_output_mae: 0.3292 - val_expression_output_accuracy: 0.4038 - val_expression_output_loss: 1.5658 - val_loss: 1.6342 - val_va_output_loss: 0.1369 - val_va_output_mae: 0.2966
Epoch 29/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 48s 480ms/step - expression_output_accuracy: 0.4748 - expression_output_loss: 1.3819 - loss: 1.4662 - va_output_loss: 0.1685 - va_output_mae: 0.3302 - val_expression_output_accuracy: 0.3675 - val_expression_output_loss: 1.6129 - val_loss: 1.6823 - val_va_output_loss: 0.1387 - val_va_output_mae: 0.3004
Epoch 30/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 48s 476ms/step - expression_output_accuracy: 0.4782 - expression_output_loss: 1.3375 - loss: 1.4223 - va_output_loss: 0.1696 - va_output_mae: 0.3299 - val_expression_output_accuracy: 0.3613 - val_expression_output_loss: 1.6543 - val_loss: 1.7231 - val_va_output_loss: 0.1376 - val_va